# LLM Inference Engine — T4 Tuned
**GPT-2 + FlashAttention (Triton) + Fused INT8 (Triton & CUDA) + Speculative Decoding (distilGPT-2)**

All hyperparameters tuned for T4. Author: Somya Padhy

In [ ]:
!pip install -q triton transformers
import torch, math, time, gc, copy
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
try:
    import triton, triton.language as tl
    HAS_TRITON = True; print(f'Triton {triton.__version__}')
except: HAS_TRITON = False
GPU = torch.cuda.get_device_name(0)
IS_T4 = 'T4' in GPU
# T4-tuned block sizes (T4 has 64KB SRAM vs A100's 192KB)
BLOCK = 32 if IS_T4 else 64
print(f'GPU: {GPU} | Block size: {BLOCK}')

In [ ]:
# ============================================================
# ALL TUNABLE HYPERPARAMETERS IN ONE PLACE
# ============================================================
SPEC_K = 3              # draft tokens per round (3 for T4, 4-5 for A100)
SPEC_TEMP = 0.3         # lower = more deterministic = higher acceptance (was 0.8)
TOP_K = 50              # top-k sampling
GEN_TEMP = 0.8          # temperature for normal generation
N_BENCH_TOKENS = 100    # tokens to generate in benchmarks
print(f'Spec K={SPEC_K}, Spec temp={SPEC_TEMP}, Block={BLOCK}')

## 1. KV Cache

In [ ]:
class KVCache:
    def __init__(self,B,H,D,max_len,device,dtype):
        self.seq_len=0
        self.k=torch.zeros(B,H,max_len,D,device=device,dtype=dtype)
        self.v=torch.zeros(B,H,max_len,D,device=device,dtype=dtype)
    def update(self,k_new,v_new):
        n=k_new.size(2)
        self.k[:,:,self.seq_len:self.seq_len+n,:]=k_new
        self.v[:,:,self.seq_len:self.seq_len+n,:]=v_new
        self.seq_len+=n
        return self.k[:,:,:self.seq_len,:],self.v[:,:,:self.seq_len,:]
    def rollback(self,n):
        self.k[:,:,self.seq_len-n:self.seq_len,:]=0
        self.v[:,:,self.seq_len-n:self.seq_len,:]=0
        self.seq_len-=n
print('KVCache ready')

## 2. FlashAttention (Triton) — BLOCK tuned per GPU

In [ ]:
if HAS_TRITON:
    @triton.jit
    def _flash_attn_fwd(
        Q,K,V,O, sq,sh,ss,sd, kq,kh,ks,kd, vq,vh,vs,vd, oq,oh,os,od,
        SQ:tl.constexpr,SKV:tl.constexpr,HD:tl.constexpr,scale,
        BQ:tl.constexpr,BKV:tl.constexpr,CAUSAL:tl.constexpr,
    ):
        bq_id=tl.program_id(0);bh=tl.program_id(1)
        qo=bh.to(tl.int64)*sh;ko=bh.to(tl.int64)*kh;vo=bh.to(tl.int64)*vh;oo=bh.to(tl.int64)*oh
        rq=bq_id*BQ+tl.arange(0,BQ);rkv=tl.arange(0,BKV);rd=tl.arange(0,HD)
        Qb=tl.load(Q+qo+rq[:,None]*ss+rd[None,:]*sd,mask=rq[:,None]<SQ,other=0.0)
        mi=tl.zeros([BQ],dtype=tl.float32)-float('inf')
        li=tl.zeros([BQ],dtype=tl.float32)
        Oa=tl.zeros([BQ,HD],dtype=tl.float32)
        kv_end=tl.minimum((bq_id+1)*BQ+(SKV-SQ),SKV) if CAUSAL else SKV
        for s in range(0,kv_end,BKV):
            ki=s+rkv
            KT=tl.load(K+ko+ki[None,:]*ks+rd[:,None]*kd,mask=ki[None,:]<SKV,other=0.0)
            QK=tl.dot(Qb,KT)*scale
            if CAUSAL:
                QK=tl.where((rq+(SKV-SQ))[:,None]>=ki[None,:],QK,float('-inf'))
            mn=tl.maximum(mi,tl.max(QK,1));P=tl.math.exp(QK-mn[:,None])
            a=tl.math.exp(mi-mn);li=li*a+tl.sum(P,1)
            Vb=tl.load(V+vo+ki[:,None]*vs+rd[None,:]*vd,mask=ki[:,None]<SKV,other=0.0)
            Oa=Oa*a[:,None];Oa=tl.dot(P.to(tl.float16),Vb,Oa);mi=mn
        Oa=Oa/li[:,None]
        tl.store(O+oo+rq[:,None]*os+rd[None,:]*od,Oa.to(tl.float16),mask=rq[:,None]<SQ)

def flash_attention(Q,K,V,causal=True,scale=None):
    B,H,Sq,D=Q.shape;Skv=K.shape[2]
    if scale is None:scale=1.0/math.sqrt(D)
    Q,K,V=Q.contiguous(),K.contiguous(),V.contiguous()
    O=torch.empty_like(Q)
    _flash_attn_fwd[(triton.cdiv(Sq,BLOCK),B*H)](
        Q,K,V,O,Q.stride(0),Q.stride(1),Q.stride(2),Q.stride(3),
        K.stride(0),K.stride(1),K.stride(2),K.stride(3),
        V.stride(0),V.stride(1),V.stride(2),V.stride(3),
        O.stride(0),O.stride(1),O.stride(2),O.stride(3),
        SQ=Sq,SKV=Skv,HD=D,scale=scale,BQ=BLOCK,BKV=BLOCK,CAUSAL=causal)
    return O
print(f'FlashAttention ready (BLOCK={BLOCK})')

## 3. Fused INT8 Matmul (Triton) — BLOCK tuned per GPU

In [ ]:
if HAS_TRITON:
    @triton.jit
    def _fused_int8_mm(X,W,S,Out,M,N,K:tl.constexpr,xm,xk,wn,wk,om,on,
                       BM:tl.constexpr,BN:tl.constexpr,BK:tl.constexpr):
        pm,pn=tl.program_id(0),tl.program_id(1)
        rm=pm*BM+tl.arange(0,BM);rn=pn*BN+tl.arange(0,BN);rk=tl.arange(0,BK)
        acc=tl.zeros([BM,BN],dtype=tl.float32)
        for k in range(0,K,BK):
            ki=k+rk
            x=tl.load(X+rm[:,None]*xm+ki[None,:]*xk,mask=(rm[:,None]<M)&(ki[None,:]<K),other=0.0)
            w=tl.load(W+rn[:,None]*wn+ki[None,:]*wk,mask=(rn[:,None]<N)&(ki[None,:]<K),other=0)
            acc+=tl.dot(x,tl.trans(w.to(tl.float16)))
        s=tl.load(S+rn,mask=rn<N,other=1.0)
        acc=acc*s[None,:].to(tl.float32)
        tl.store(Out+rm[:,None]*om+rn[None,:]*on,acc.to(tl.float16),mask=(rm[:,None]<M)&(rn[None,:]<N))

def fused_int8_matmul(x,w8,scale):
    x2=x.reshape(-1,x.shape[-1]);M,K=x2.shape;N=w8.shape[0]
    out=torch.empty(M,N,device=x.device,dtype=x.dtype)
    _fused_int8_mm[(triton.cdiv(M,BLOCK),triton.cdiv(N,BLOCK))](
        x2,w8,scale,out,M,N,K,x2.stride(0),x2.stride(1),w8.stride(0),w8.stride(1),
        out.stride(0),out.stride(1),BM=BLOCK,BN=BLOCK,BK=BLOCK)
    return out.reshape(*x.shape[:-1],N)
print(f'Fused INT8 ready (BLOCK={BLOCK})')

## 4. CUDA INT8 Matmul

In [ ]:
from torch.utils.cpp_extension import load_inline
import os,shutil
for d in ['int8mm','int8mm2','int8mm3']:
    p=os.path.expanduser(f'~/.cache/torch_extensions/py312_cu128/{d}')
    if os.path.exists(p):shutil.rmtree(p)
cuda_src=r"""
#include <torch/extension.h>
#include <cuda_fp16.h>
__global__ void k(const half*X,const int8_t*W,const half*S,half*O,int M,int N,int K){
    int r=blockIdx.y*blockDim.y+threadIdx.y,c=blockIdx.x*blockDim.x+threadIdx.x;
    if(r<M&&c<N){float s=0;for(int k=0;k<K;k++)s+=__half2float(X[r*K+k])*(float)W[c*K+k];
    O[r*N+c]=__float2half(s*__half2float(S[c]));}
}
torch::Tensor int8mm(torch::Tensor X,torch::Tensor W,torch::Tensor S){
    int M=X.size(0),K=X.size(1),N=W.size(0);auto O=torch::empty({M,N},X.options());
    dim3 b(16,16),g((N+15)/16,(M+15)/16);
    k<<<g,b>>>((half*)X.data_ptr(),(int8_t*)W.data_ptr(),(half*)S.data_ptr(),(half*)O.data_ptr(),M,N,K);
    return O;}
"""
try:
    cuda_mod=load_inline(name='int8mm3',cpp_sources=['torch::Tensor int8mm(torch::Tensor,torch::Tensor,torch::Tensor);'],
        cuda_sources=[cuda_src],functions=['int8mm'],verbose=False,extra_cuda_cflags=['-O3'])
    HAS_CUDA=True;print('CUDA INT8 compiled')
except Exception as e:HAS_CUDA=False;print(f'CUDA failed: {e}')

## 5. Quantization

In [ ]:
def quantize_int8(w):
    s=w.abs().amax(dim=1,keepdim=True)/127.0;s=torch.clamp(s,min=1e-10)
    return torch.clamp(torch.round(w/s),-127,127).to(torch.int8),s.squeeze(1).half()

class FusedQLinear(nn.Module):
    def __init__(self,inf,outf,w8,sc,bias=None):
        super().__init__()
        self.register_buffer('w8',w8);self.register_buffer('sc',sc)
        self.bias=nn.Parameter(bias) if bias is not None else None
    def forward(self,x):
        o=fused_int8_matmul(x,self.w8,self.sc)
        return o+self.bias if self.bias is not None else o
    @classmethod
    def from_linear(cls,lin):
        w8,sc=quantize_int8(lin.weight.data.float())
        b=lin.bias.data.clone() if lin.bias is not None else None
        return cls(lin.in_features,lin.out_features,w8.to(lin.weight.device),sc.to(lin.weight.device),b)

class NaiveQLinear(nn.Module):
    def __init__(self,inf,outf,w8,sc,bias=None):
        super().__init__()
        self.register_buffer('w8',w8);self.register_buffer('sc',sc)
        self.bias=nn.Parameter(bias) if bias is not None else None
    def forward(self,x):
        w=(self.w8.float()*self.sc.float().unsqueeze(1)).to(x.dtype)
        return F.linear(x,w,self.bias)
    @classmethod
    def from_linear(cls,lin):
        w8,sc=quantize_int8(lin.weight.data.float())
        b=lin.bias.data.clone() if lin.bias is not None else None
        return cls(lin.in_features,lin.out_features,w8.to(lin.weight.device),sc.to(lin.weight.device),b)

def quantize_model(model,method='fused',skip=('lm_head',)):
    Cls=FusedQLinear if method=='fused' else NaiveQLinear
    c=0
    for name,mod in list(model.named_modules()):
        if isinstance(mod,nn.Linear) and not any(s in name for s in skip):
            parts=name.rsplit('.',1)
            parent=dict(model.named_modules())[parts[0]] if len(parts)>1 else model
            setattr(parent,parts[-1],Cls.from_linear(mod));c+=1
    gc.collect();torch.cuda.empty_cache()
    print(f'  Quantized {c} layers ({method}) -- {torch.cuda.memory_allocated()/1e6:.0f} MB')
    return model
print('Quantization ready')

## 6. GPT-2 (from scratch)

In [ ]:
@dataclass
class GPT2Config:
    vocab_size:int=50257;seq_len:int=1024;embed_dim:int=768
    num_heads:int=12;num_layers:int=12;bias:bool=True

class Attention(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.nh=c.num_heads;self.d=c.embed_dim;self.hd=self.d//self.nh
        self.c_attn=nn.Linear(self.d,3*self.d,bias=c.bias)
        self.c_proj=nn.Linear(self.d,self.d,bias=c.bias)
        self.scale=1.0/math.sqrt(self.hd)
    def forward(self,x,kv_cache=None,use_flash=False):
        B,T,C=x.shape
        qkv=self.c_attn(x);q,k,v=qkv.split(self.d,dim=2)
        q=q.view(B,T,self.nh,self.hd).transpose(1,2)
        k=k.view(B,T,self.nh,self.hd).transpose(1,2)
        v=v.view(B,T,self.nh,self.hd).transpose(1,2)
        if kv_cache is not None:k,v=kv_cache.update(k,v)
        if use_flash and HAS_TRITON:
            out=flash_attention(q,k,v,causal=True,scale=self.scale)
        else:
            Sq,Sk=q.size(2),k.size(2)
            att=torch.matmul(q,k.transpose(-2,-1))*self.scale
            att=att.masked_fill(torch.triu(torch.ones(Sq,Sk,device=x.device,dtype=torch.bool),diagonal=Sk-Sq+1),float('-inf'))
            att=F.softmax(att.float(),dim=-1).to(q.dtype)
            out=torch.matmul(att,v)
        return self.c_proj(out.transpose(1,2).contiguous().view(B,T,C))

class MLP(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.c_fc=nn.Linear(c.embed_dim,4*c.embed_dim,bias=c.bias)
        self.c_proj=nn.Linear(4*c.embed_dim,c.embed_dim,bias=c.bias)
        self.act=nn.GELU(approximate='tanh')
    def forward(self,x):return self.c_proj(self.act(self.c_fc(x)))

class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.ln_1=nn.LayerNorm(c.embed_dim);self.attn=Attention(c)
        self.ln_2=nn.LayerNorm(c.embed_dim);self.mlp=MLP(c)
    def forward(self,x,kv_cache=None,use_flash=False):
        x=x+self.attn(self.ln_1(x),kv_cache,use_flash)
        return x+self.mlp(self.ln_2(x))

class GPT2(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.config=config
        self.wte=nn.Embedding(config.vocab_size,config.embed_dim)
        self.wpe=nn.Embedding(config.seq_len,config.embed_dim)
        self.blocks=nn.ModuleList([Block(config) for _ in range(config.num_layers)])
        self.ln_f=nn.LayerNorm(config.embed_dim)
        self.lm_head=nn.Linear(config.embed_dim,config.vocab_size,bias=False)
        self.lm_head.weight=self.wte.weight
        print(f'  GPT-2: {sum(p.numel() for p in self.parameters())/1e6:.1f}M ({config.num_layers}L)')
    def forward(self,ids,kv_caches=None,use_flash=False):
        B,T=ids.shape
        off=kv_caches[0].seq_len if(kv_caches and kv_caches[0].seq_len>0)else 0
        pos=torch.arange(off,off+T,device=ids.device).unsqueeze(0).expand(B,-1)
        x=self.wte(ids)+self.wpe(pos)
        for i,b in enumerate(self.blocks):
            x=b(x,kv_caches[i] if kv_caches else None,use_flash)
        return self.lm_head(self.ln_f(x))
    def create_kv_caches(self,B=1,max_len=1024):
        d,dt=next(self.parameters()).device,next(self.parameters()).dtype
        hd=self.config.embed_dim//self.config.num_heads
        return [KVCache(B,self.config.num_heads,hd,max_len,d,dt) for _ in range(self.config.num_layers)]
    @classmethod
    def from_pretrained(cls,name='gpt2',device='cuda'):
        from transformers import GPT2LMHeadModel
        print(f'Loading {name}...')
        nl={'gpt2':12,'distilgpt2':6}[name];config=GPT2Config(num_layers=nl)
        model=cls(config);hf=GPT2LMHeadModel.from_pretrained(name);hf_sd=hf.state_dict()
        km={}
        for i in range(nl):
            p=f'transformer.h.{i}'
            for s,d in [('ln_1.weight','ln_1.weight'),('ln_1.bias','ln_1.bias'),
                ('attn.c_attn.weight','attn.c_attn.weight'),('attn.c_attn.bias','attn.c_attn.bias'),
                ('attn.c_proj.weight','attn.c_proj.weight'),('attn.c_proj.bias','attn.c_proj.bias'),
                ('ln_2.weight','ln_2.weight'),('ln_2.bias','ln_2.bias'),
                ('mlp.c_fc.weight','mlp.c_fc.weight'),('mlp.c_fc.bias','mlp.c_fc.bias'),
                ('mlp.c_proj.weight','mlp.c_proj.weight'),('mlp.c_proj.bias','mlp.c_proj.bias')]:
                km[f'{p}.{s}']=f'blocks.{i}.{d}'
        km.update({'transformer.wte.weight':'wte.weight','transformer.wpe.weight':'wpe.weight',
            'transformer.ln_f.weight':'ln_f.weight','transformer.ln_f.bias':'ln_f.bias'})
        new_sd={}
        for hk,ok in km.items():
            if hk not in hf_sd:continue
            param=hf_sd[hk]
            if 'weight' in hk and param.dim()==2 and 'ln' not in hk and 'wte' not in hk and 'wpe' not in hk:
                if param.shape!=model.state_dict()[ok].shape:param=param.t()
            new_sd[ok]=param
        model.load_state_dict(new_sd,strict=False)
        del hf,hf_sd;gc.collect();torch.cuda.empty_cache()
        model=model.to(device).half().eval()
        print(f'  Loaded ({torch.cuda.memory_allocated()/1e6:.0f} MB)')
        return model
print('GPT-2 defined')

## 7. Generation + Speculative Decoding (carry-aligned, verbose)

In [ ]:
def sample_top_k(logits,k=TOP_K,temperature=1.0):
    if temperature!=1.0:logits=logits/temperature
    vals,idxs=torch.topk(logits,k,dim=-1)
    return torch.gather(idxs,-1,torch.multinomial(F.softmax(vals,dim=-1),1))

@torch.no_grad()
def generate(model,input_ids,max_new=100,temperature=GEN_TEMP,use_flash=False):
    B,T=input_ids.shape;kv=model.create_kv_caches(B,T+max_new)
    nl=model(input_ids,kv_caches=kv,use_flash=use_flash)[:,-1,:]
    tokens=[]
    for _ in range(max_new):
        tok=sample_top_k(nl,temperature=temperature);tokens.append(tok)
        if(tok==50256).all():break
        nl=model(tok,kv_caches=kv,use_flash=use_flash)[:,-1,:]
    return torch.cat([input_ids]+tokens,dim=1)

def spec_sample(tl,dl,dt,temperature,tokenizer=None,verbose=False,round_num=0):
    B,K,V=dl.shape
    if temperature!=1.0:tl,dl=tl/temperature,dl/temperature
    p=F.softmax(tl[:,:K,:],dim=-1);q=F.softmax(dl,dim=-1)
    idx=dt.unsqueeze(-1)
    p_tok=torch.gather(p,-1,idx).squeeze(-1)
    q_tok=torch.gather(q,-1,idx).squeeze(-1)
    ratio=torch.clamp(p_tok/(q_tok+1e-10),max=1.0)
    if verbose and tokenizer:
        print(f'  Round {round_num}:')
        for i in range(K):
            tok_s=tokenizer.decode([dt[0,i].item()])
            t_top=tokenizer.decode([tl[0,i,:].argmax().item()])
            d_top=tokenizer.decode([dl[0,i,:].argmax().item()])
            print(f'    [{i}] draft="{tok_s}"  p_t={p_tok[0,i].item():.3f}  p_d={q_tok[0,i].item():.3f}'
                  f'  ratio={ratio[0,i].item():.3f}  t_top="{t_top}" d_top="{d_top}"'
                  f'  {"AGREE" if tl[0,i,:].argmax()==dl[0,i,:].argmax() else "DIFFER"}')
    rng=torch.rand_like(ratio)
    rej=(~(rng<ratio))[0].nonzero(as_tuple=True)[0]
    if len(rej)==0:
        bonus=sample_top_k(tl[:,K,:],temperature=temperature)
        if verbose and tokenizer:
            print(f'    => ALL {K} ACCEPTED + bonus="{tokenizer.decode([bonus[0,0].item()])}"')
        return torch.cat([dt,bonus],dim=1),K
    n=rej[0].item()
    adj=torch.clamp(p[:,n,:]-q[:,n,:],min=0.0)
    rs=torch.multinomial(adj/(adj.sum(-1,keepdim=True)+1e-10),1)
    if verbose and tokenizer:
        print(f'    => REJECTED at [{n}], kept {n}/{K}, resample="{tokenizer.decode([rs[0,0].item()])}"')
    return(torch.cat([dt[:,:n],rs],dim=1) if n>0 else rs),n

@torch.no_grad()
def speculative_generate(target,draft,input_ids,max_new=100,K=SPEC_K,
                         temperature=SPEC_TEMP,use_flash=False,verbose=False,tokenizer=None):
    B,T=input_ids.shape;ml=T+max_new+K+10
    tkv,dkv=target.create_kv_caches(B,ml),draft.create_kv_caches(B,ml)
    if verbose:print(f'\nSPECULATIVE TRACE: K={K}, temp={temperature}')

    # Prefill
    t_pf=target(input_ids,kv_caches=tkv,use_flash=use_flash)
    d_pf=draft(input_ids,kv_caches=dkv,use_flash=use_flash)
    t_carry=t_pf[:,-1:,:]  # (B,1,V)
    d_carry=d_pf[:,-1,:]   # (B,V)

    if verbose and tokenizer:
        t_top=tokenizer.decode([t_carry[0,0].argmax().item()])
        d_top=tokenizer.decode([d_carry[0].argmax().item()])
        print(f'  After prefill: target predicts "{t_top}", draft predicts "{d_top}" ({"AGREE" if t_carry[0,0].argmax()==d_carry[0].argmax() else "DIFFER"})')

    gen,td,ta,tc=[],0,0,0
    while len(gen)<max_new:
        # Draft proposes K tokens
        dtoks,dlogs=[],[]
        cur=d_carry
        for _ in range(K):
            dlogs.append(cur)
            tok=sample_top_k(cur,temperature=temperature);dtoks.append(tok)
            cur=draft(tok,kv_caches=dkv,use_flash=use_flash)[:,-1,:]
        dt=torch.cat(dtoks,dim=1);dl=torch.stack(dlogs,dim=1)
        td+=K

        # Target verifies
        tl_out=target(dt,kv_caches=tkv,use_flash=use_flash)  # (B,K,V)
        tc+=1
        full=torch.cat([t_carry,tl_out],dim=1)  # (B,K+1,V) aligned

        # Accept/reject
        do_verbose=verbose and tc<=5
        accepted,na=spec_sample(full,dl,dt,temperature,
                                tokenizer=tokenizer if do_verbose else None,
                                verbose=do_verbose,round_num=tc)
        ta+=na

        # Rollback + get new carry
        nr=K-na
        if nr>0:
            for c in tkv:c.rollback(nr)
            for c in dkv:c.rollback(nr)
            last=accepted[:,-1:]
            t_carry=target(last,kv_caches=tkv,use_flash=use_flash)[:,-1:,:]
            d_carry=draft(last,kv_caches=dkv,use_flash=use_flash)[:,-1,:]
        else:
            bonus=accepted[:,-1:]
            t_carry=target(bonus,kv_caches=tkv,use_flash=use_flash)[:,-1:,:]
            d_carry=draft(bonus,kv_caches=dkv,use_flash=use_flash)[:,-1,:]

        for i in range(accepted.size(1)):
            gen.append(accepted[:,i:i+1])
            if len(gen)>=max_new:break

    rate=ta/td if td else 0;tpc=len(gen)/tc if tc else 0
    print(f'  Spec: {ta}/{td} accepted ({rate:.0%}), {tpc:.1f} tok/call, {tc} target calls for {len(gen)} tokens')
    return torch.cat([input_ids]+gen,dim=1)
print(f'Generation ready (K={SPEC_K}, temp={SPEC_TEMP})')

## 8. Load Models

In [ ]:
from transformers import GPT2Tokenizer
tokenizer=GPT2Tokenizer.from_pretrained('gpt2')
model=GPT2.from_pretrained('gpt2','cuda')
draft=GPT2.from_pretrained('distilgpt2','cuda')
prompt='The future of artificial intelligence is'
ids=tokenizer.encode(prompt,return_tensors='pt').to('cuda')
print(f'\nTarget: GPT-2 124M (12L) | Draft: distilGPT-2 82M (6L)')
print(f'Prompt: "{prompt}" ({ids.shape[1]} tokens)')

## 9. Test Generation (verify both models produce coherent text)

In [ ]:
print('--- Target (standard attn) ---')
print(tokenizer.decode(generate(model,ids,60)[0],skip_special_tokens=True))
print('\n--- Target (FlashAttention) ---')
print(tokenizer.decode(generate(model,ids,60,use_flash=True)[0],skip_special_tokens=True))
print('\n--- Draft (distilGPT-2) ---')
print(tokenizer.decode(generate(draft,ids,60)[0],skip_special_tokens=True))

## 10. Speculative Decoding — VERBOSE TRACE (5 rounds)
Shows per-token: draft choice, target/draft probabilities, acceptance ratio, top-1 agreement

In [ ]:
out=speculative_generate(model,draft,ids,40,K=SPEC_K,temperature=SPEC_TEMP,
                         verbose=True,tokenizer=tokenizer)
print(f'\nOutput: "{tokenizer.decode(out[0],skip_special_tokens=True)}"')

## 11. Temperature Sweep — find the sweet spot

In [ ]:
print('='*60)
print('TEMPERATURE SWEEP (K=3)')
print('='*60)
for temp in [0.1, 0.2, 0.3, 0.5, 0.8, 1.0]:
    print(f'\ntemp={temp}:')
    speculative_generate(model,draft,ids,100,K=3,temperature=temp)
print('='*60)

## 12. K Sweep — find optimal draft length

In [ ]:
print('='*60)
print(f'K SWEEP (temp={SPEC_TEMP})')
print('='*60)
for K in [1,2,3,4,5,6]:
    print(f'\nK={K}:')
    speculative_generate(model,draft,ids,100,K=K,temperature=SPEC_TEMP)
print('='*60)

## 13. Prefill Benchmark (where FlashAttention actually wins)
FlashAttention helps during PREFILL (long prompts), not single-token decode.

In [ ]:
def naive_attn(Q,K,V):
    sc=1.0/math.sqrt(Q.shape[-1]);Sq,Sk=Q.size(2),K.size(2)
    att=torch.matmul(Q,K.transpose(-2,-1))*sc
    att=att.masked_fill(torch.triu(torch.ones(Sq,Sk,device=Q.device,dtype=torch.bool),diagonal=Sk-Sq+1),float('-inf'))
    return torch.matmul(F.softmax(att.float(),dim=-1).to(Q.dtype),V)

print('='*70)
print(f'FLASHATTENTION PREFILL -- {GPU}')
print('='*70)
print(f'{"B,H,S,D":<20} {"Naive":>10} {"Triton":>10} {"Speedup":>10}')
for S in [64,128,256,512,1024]:
    B,H,D=1,12,64
    Q=torch.randn(B,H,S,D,device='cuda',dtype=torch.float16)
    K_=torch.randn(B,H,S,D,device='cuda',dtype=torch.float16)
    V_=torch.randn(B,H,S,D,device='cuda',dtype=torch.float16)
    sc=1.0/math.sqrt(D)
    for _ in range(5):naive_attn(Q,K_,V_);flash_attention(Q,K_,V_,causal=True,scale=sc)
    torch.cuda.synchronize()
    t0=time.perf_counter()
    for _ in range(50):naive_attn(Q,K_,V_)
    torch.cuda.synchronize();nm=(time.perf_counter()-t0)/50*1000
    t0=time.perf_counter()
    for _ in range(50):flash_attention(Q,K_,V_,causal=True,scale=sc)
    torch.cuda.synchronize();tm=(time.perf_counter()-t0)/50*1000
    print(f'({B},{H},{S},{D}){"":>8} {nm:>8.2f}ms {tm:>8.2f}ms {nm/tm:>8.2f}x')

print('\n--- Full model prefill (not just kernel) ---')
print(f'{"Seq len":<12} {"Standard":>12} {"Flash":>12} {"Speedup":>10}')
for seq_len in [64,128,256,512]:
    dummy=torch.randint(0,50257,(1,seq_len),device='cuda')
    for _ in range(3):
        kv=model.create_kv_caches(1,seq_len+10);model(dummy,kv_caches=kv,use_flash=False)
        kv=model.create_kv_caches(1,seq_len+10);model(dummy,kv_caches=kv,use_flash=True)
    torch.cuda.synchronize()
    t0=time.perf_counter()
    for _ in range(20):kv=model.create_kv_caches(1,seq_len+10);model(dummy,kv_caches=kv,use_flash=False)
    torch.cuda.synchronize();std=(time.perf_counter()-t0)/20*1000
    t0=time.perf_counter()
    for _ in range(20):kv=model.create_kv_caches(1,seq_len+10);model(dummy,kv_caches=kv,use_flash=True)
    torch.cuda.synchronize();fa=(time.perf_counter()-t0)/20*1000
    print(f'{seq_len:<12} {std:>10.2f}ms {fa:>10.2f}ms {std/fa:>8.2f}x')
print('='*70)

## 14. Fused INT8 Kernel Benchmark

In [ ]:
print('='*70)
print(f'FUSED INT8 MATMUL -- {GPU} (BLOCK={BLOCK})')
print('='*70)
hdr=f'{"M,N,K":<20} {"Naive":>10} {"Fused":>10}'
if HAS_CUDA:hdr+=f' {"CUDA":>10}'
hdr+=f' {"Speedup":>10}'
print(hdr)
for M,N,K in [(1,768,768),(1,3072,768),(8,768,768),(8,3072,768),(32,768,768),(32,3072,768),(128,768,768),(128,3072,768)]:
    x=torch.randn(M,K,device='cuda',dtype=torch.float16)
    W=torch.randn(N,K,device='cuda');w8,sc=quantize_int8(W);w8,sc=w8.cuda(),sc.cuda()
    for _ in range(5):F.linear(x,(w8.float()*sc.float().unsqueeze(1)).half());fused_int8_matmul(x,w8,sc)
    torch.cuda.synchronize()
    t0=time.perf_counter()
    for _ in range(50):F.linear(x,(w8.float()*sc.float().unsqueeze(1)).half())
    torch.cuda.synchronize();nm=(time.perf_counter()-t0)/50*1000
    t0=time.perf_counter()
    for _ in range(50):fused_int8_matmul(x,w8,sc)
    torch.cuda.synchronize();tm=(time.perf_counter()-t0)/50*1000
    row=f'({M},{N},{K}){"":>8} {nm:>8.3f}ms {tm:>8.3f}ms'
    if HAS_CUDA:
        for _ in range(5):cuda_mod.int8mm(x,w8,sc)
        torch.cuda.synchronize()
        t0=time.perf_counter()
        for _ in range(50):cuda_mod.int8mm(x,w8,sc)
        torch.cuda.synchronize();cm=(time.perf_counter()-t0)/50*1000
        row+=f' {cm:>8.3f}ms'
    row+=f' {nm/tm:>8.2f}x'
    print(row)
print('='*70)

## 15. VRAM: FP32 vs FP16 vs INT8

In [ ]:
print('='*60)
print(f'VRAM -- {GPU}')
print('='*60)
gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
from transformers import GPT2LMHeadModel
hf32=GPT2LMHeadModel.from_pretrained('gpt2').float().to('cuda')
fp32_mem=torch.cuda.memory_allocated()/1e6
del hf32;gc.collect();torch.cuda.empty_cache()
print(f'  FP32: {fp32_mem:.0f} MB')
torch.cuda.reset_peak_memory_stats()
m16=GPT2.from_pretrained('gpt2','cuda')
fp16_mem=torch.cuda.memory_allocated()/1e6
print(f'  FP16: {fp16_mem:.0f} MB')
quantize_model(m16,method='fused')
int8_mem=torch.cuda.memory_allocated()/1e6
print(f'  INT8: {int8_mem:.0f} MB')
del m16;gc.collect();torch.cuda.empty_cache()
print(f'\n  FP32->INT8: {fp32_mem/int8_mem:.1f}x')
print('='*60)

## 16. End-to-End Benchmark (with tuned params)

In [ ]:
def bench(mdl,label,use_flash=False,draft_mdl=None,runs=3):
    gfn=lambda:(speculative_generate(mdl,draft_mdl,ids,N_BENCH_TOKENS,K=SPEC_K,
                temperature=SPEC_TEMP,use_flash=use_flash) if draft_mdl
                else generate(mdl,ids,N_BENCH_TOKENS,use_flash=use_flash))
    gfn();torch.cuda.synchronize()
    times=[]
    for _ in range(runs):
        torch.cuda.synchronize();t0=time.perf_counter()
        out=gfn();torch.cuda.synchronize()
        times.append((out.shape[1]-ids.shape[1])/(time.perf_counter()-t0))
    avg=sum(times)/len(times);mem=torch.cuda.max_memory_allocated()/1e6
    print(f'  {label:<50} {avg:>7.1f} tok/s {mem:>7.0f} MB');return avg

print('\n'+'='*75)
print(f'END-TO-END -- {GPU} -- GPT-2 124M -- {N_BENCH_TOKENS} tokens')
print(f'Spec params: K={SPEC_K}, temp={SPEC_TEMP}')
print('='*75)
print(f'  {"Config":<50} {"Speed":>7} {"Memory":>7}')
print(f'  {"-"*50} {"-"*7} {"-"*7}')

torch.cuda.reset_peak_memory_stats()
base=bench(model,'Baseline (FP16, standard attn)')

torch.cuda.reset_peak_memory_stats()
try:
    fa=bench(model,'+ FlashAttention (Triton)',use_flash=True)
    print(f'    -> {fa/base:.2f}x')
except Exception as e:print(f'  FA FAILED: {e}')

torch.cuda.reset_peak_memory_stats()
sp=bench(model,f'+ Speculative (distilGPT-2, K={SPEC_K}, temp={SPEC_TEMP})',draft_mdl=draft)
print(f'    -> {sp/base:.2f}x')

torch.cuda.reset_peak_memory_stats()
spfa=bench(model,'+ Speculative + FlashAttention',use_flash=True,draft_mdl=draft)
print(f'    -> {spfa/base:.2f}x')

del model,draft;gc.collect();torch.cuda.empty_cache()
mf=GPT2.from_pretrained('gpt2','cuda');quantize_model(mf,method='fused')
torch.cuda.reset_peak_memory_stats()
qf=bench(mf,'+ INT8 Fused Triton')
print(f'    -> {qf/base:.2f}x')

del mf;gc.collect();torch.cuda.empty_cache()
mn=GPT2.from_pretrained('gpt2','cuda');quantize_model(mn,method='naive')
torch.cuda.reset_peak_memory_stats()
qn=bench(mn,'+ INT8 Naive (comparison)')
print(f'    -> {qn/base:.2f}x | Fused is {qf/qn:.2f}x faster than Naive')
print('='*75)

## 17. Summary

In [ ]:
best=max(sp/base,spfa/base)
print('\n'+'='*75)
print(f'RESULTS -- {GPU}')
print('='*75)
print(f'  VRAM: FP32 ({fp32_mem:.0f} MB) -> INT8 ({int8_mem:.0f} MB) = {fp32_mem/int8_mem:.1f}x')
print(f'  Speculative: {sp/base:.2f}x (K={SPEC_K}, temp={SPEC_TEMP})')
print(f'  Spec+Flash:  {spfa/base:.2f}x')
print(f'  Fused INT8:  {qf/qn:.2f}x faster than naive')
print(f'  FlashAttn:   see prefill benchmark above')
print('='*75)